<a href="https://colab.research.google.com/github/SebaAcunaC/YOLO11-Vigilancia-Urbana-USACH/blob/main/Semana-3-Comparativa/Semana3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# PROYECTO YOLO11 - SEMANA 3: COMPARATIVA TÉCNICA (YOLO11 vs YOLOv8)
# Objetivo: Validar la superioridad de YOLO11 en precisión y convergencia.
# ==============================================================================

# 1. SETUP E INSTALACIÓN
!pip install ultralytics roboflow pandas matplotlib -q

import os
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from roboflow import Roboflow

# 2. DESCARGA DEL DATASET (REPRODUCIBILIDAD)
api_key = input("Introduce tu API Key de Roboflow: ")
rf = Roboflow(api_key=api_key)
project = rf.workspace("sebastianacua").project("person-hgivm-vyoba")
dataset = project.version(1).download("yolov8")

# Configuración automática del data.yaml para evitar errores de ruta
data_yaml_path = os.path.join(dataset.location, "data.yaml")
with open(data_yaml_path, 'r') as f:
    config = yaml.safe_load(f)
config['train'] = os.path.join(dataset.location, "train/images")
config['val'] = os.path.join(dataset.location, "valid/images")
with open(data_yaml_path, 'w') as f:
    yaml.dump(config, f)

# 3. ENTRENAMIENTO "ESPEJO" (YOLOv8 Nano)
# Entrenamos la versión anterior para comparar con los resultados de la Semana 2
model_v8 = YOLO('yolov8n.pt')
results_v8 = model_v8.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    name='entrenamiento_personas_v8',
    project='runs/comparativa' # Guardado local para GitHub
)

# 4. CARGA DE RESULTADOS (YOLO11 ya debe haber sido entrenado o cargar su CSV)
# Nota: Para la gráfica, usaremos el CSV generado por YOLOv8 y el de YOLO11
path_v8 = 'runs/comparativa/entrenamiento_personas_v8/results.csv'
# Si tienes el CSV de la semana 2 en GitHub, puedes descargarlo aquí con !wget
# Por ahora, simularemos la comparativa si ambos están en el entorno local.

if os.path.exists(path_v8):
    res_v8 = pd.read_csv(path_v8)
    res_v8.columns = res_v8.columns.str.strip()

    # 5. VISUALIZACIÓN PROFESIONAL (Para el Mini-Paper)
    fig, ax = plt.subplots(1, 2, figsize=(16, 6))
    plt.suptitle('Análisis Comparativo: YOLO11 vs YOLOv8', fontsize=16, fontweight='bold')

    # Gráfica A: Precisión mAP50
    ax[0].plot(res_v8['metrics/mAP50(B)'], label='YOLOv8', color='#ff7f0e', linestyle='--')
    # Aquí añadirías la línea de YOLO11: ax[0].plot(res_v11['metrics/mAP50(B)'], label='YOLO11')
    ax[0].set_title('Evolución de la Precisión (mAP50)')
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)

    plt.savefig('grafica_comparativa_final.png', dpi=300)
    plt.show()